# H&M LightGCN — seed51 / M1~M5_D 8모형

**학습 순서: M1 → M2 → M3 → M4 → M5_A → M5_B → M5_C → M5_D.**
seed49·50과 같은 설계·평가 조건으로 seed51만 실행합니다. seed50 결과가 끝날 때까지 기다릴 필요는 없습니다. **다른 런타임의 H&M/GAT 학습은 중단하지 마세요.** 동일 seed51 노트북을 여러 GPU에서 동시에 실행하지 마세요.

### 고정 조건
- 신규 상품 추천: 학습 구매쌍을 평가 정답·후보에서 제외, MIN_ITEM_INTER=1.
- train 2018-09-20~2020-09-15 / test 2020-09-16~22. CLV 입력창은 최근 학습365일.
- 300epoch 마지막점만 test 평가. 검증·holdout·조기종료·중간test 없음.
- ID64, 2층, batch131072, Adam lr0.0005, ID L2 .001, M2 공유 L2 1e-6, uniform 음성1개.
- 모델마다 새 seed51 초기화와 하나의 optimizer. 다른 모델·시드 checkpoint로 초기화하지 않음.
- M3 목표 가중 CV=.20, M4 강도=.5, M2 설계·초기화 규칙 모두 유지.

### 실행 비용 절감
입력/특징/그래프를 한 번 준비하고, 고정 학습 입력을 장치에 보관합니다(추가 메모리 약0.55GiB 이하). 음성 **추첨과 재추첨은 그대로** 두고 최종 관측쌍 검사만 이진검색합니다. 로그의 GPU 값 읽기는 epoch당 한 번으로 묶고, 출력은 첫3epoch·이후5epoch마다 표시합니다. 첫3epoch의 첫5배치를 계측합니다.

**매 epoch Drive 저장은 유지**합니다. 로컬 직렬화 후 Drive 임시파일 복사·교체로 저장하고, 중복 Drive 재읽기를 줄입니다. 저장 실패·손상 checkpoint는 보존 후 중단합니다. 임베딩은 매 배치 재계산하며 AMP/CSR/compile·배치 변경·샘플 축소를 적용하지 않습니다. 실제 단축률은 GPU 계측 전에는 확정할 수 없습니다.

순열 대조·다른 시드·자동 튜닝은 실행하지 않습니다. 전체142지표와 고객군별/노출 결과는 유지합니다. 본 파일은 **seed51 단일 결과**이며 3시드 평균/유의성은 49·50·51 원본 취합 후 별도 분석합니다. 최종3시드 축소는 교수님과 별도 합의할 사항입니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 기존 조건과 같은 L4로 실행하세요.'
SOURCE_COMMIT = '8ca3baa90be1e0ac02554e1c6fe4c1c6b8deb229'
REPO = Path('/content/clv-hm-final8-s51-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)


## 1. 고정 설계 원본 확인 — 아직 데이터 전체 로드·학습 없음

기본 참조는 기존 H&M M2 결과 폴더와 M1/M3/M4/M5_C 결과 폴더입니다. seed49 원본은 **입력·설계 지문 검사에만** 사용합니다. seed50 완료 결과는 필요하지 않습니다.

원본 보고서를 찾지 못하면 아래 주석의 보관 ZIP 경로를 지정하세요. ZIP을 새로 생성하거나 모델을 재학습할 필요는 없습니다. 참조 경로·출력 경로·코드 pin을 재개 시 그대로 유지하세요.

새 결과: `/content/drive/MyDrive/논문/data/results_v3_hm_final8_lastweek_seed51_v1`. 기존 seed49/50 폴더에는 쓰지 않습니다.

In [ ]:
import clv_hm_final8_seed51 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/clv_m2_adaptive_hm_seed49_lastweek_results.zip',
    # m4_reference='/content/drive/MyDrive/clv_hm_m4c_lastweek_seed49_results.zip',
)
print('M2 설계 원본:', cfg.m2_reference)
print('M3/M4 설계 원본:', cfg.m4_reference)
print('새 결과/체크포인트:', cfg.out_dir)
refs = screen.load_references(cfg)
print('참조 지문 확인 완료 / 학습 순서:', screen.ORDER)


## 2. 공통 입력 한 번 준비

M2 특징·M4 가중치와 M3 그래프 준비 시간을 표시합니다. 기존 원본과 입력·분할·특징·가중치가 다르면 학습 전에 중단합니다. 준비 함수는 모델을 학습하거나 test를 평가하지 않습니다.

재연결 후에는 같은 노트북의 준비 셀까지 다시 실행하세요. 입력은 다시 준비하지만, 완료 모델은 건너뛰고 미완료 모델은 저장 epoch에서 재개합니다. `loss_w`가 생략된 공통 데이터도 정상 처리합니다.

In [ ]:
prepared = screen.prepare(cfg)
assert cfg.seeds == (51,) and cfg.epochs == 300
assert prepared['data'].get('loss_w') is None
assert set(prepared['data']['splits']) == {'test'}
print('실행 폴더:', prepared['run_dir'])
print('사전점검 완료. 다음 셀에서만 8모형 학습을 시작합니다.')


## 3. 8모형 순차 학습 / 매 epoch 저장 / 중단 재개

각 모델의 300epoch 마지막점 평가와 원본 JSON 저장을 마친 뒤 다음 모델로 넘어갑니다. 완료 JSON은 재학습·재평가하지 않습니다. 중단 시 모델·Adam·난수 상태를 복원하고 마지막 저장 다음 epoch부터 재개합니다. 저장 전 중단된 epoch의 일부 계산은 다시 합니다.

첫3epoch에는 입력·음성추출과 학습계산 시간을 따로 표시합니다. 이후 로그에는 **학습 시간과 Drive 저장 시간**이 표시됩니다. 로그 사이에도 학습·매epoch 저장은 계속됩니다. 모델 간 속도가 다르므로 M1 소요시간만으로 전체8모형 시간을 단정하지 마세요.

학습 중 다른 셀을 실행하지 마세요. 자동 런타임 재연결은 지원하지 않습니다. Drive 오류가 나면 정상 재연결 후 같은 경로에서 재실행하세요. 저장본을 삭제하거나 다른 경로로 바꿔 처음부터 시작하지 마세요.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
assert result['complete'], '아직 8모형이 모두 완료되지 않았습니다.'


## 4. 완료 결과와 시간 확인 — 추가 학습·평가 없음

미완료 모델은 빈칸이며 실패 판정이 아닙니다. 화면에는 핵심12지표, CSV에는 전체142지표·모든 모델간 비교·진단을 저장합니다. 상승/하락은 통계적 유의성과 다릅니다.

남은 시간 추정은 **해당 모델의 최근5epoch 학습+저장 속도**에 한정하며, 최종평가·다음 모델·런타임 중단 시간은 포함하지 않습니다. 값이 0이어도 최종평가가 남아 있을 수 있습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
comparison = result['comparison']
vs_m1 = comparison[comparison['reference'].eq('M1')]
if not vs_m1.empty:
    display(vs_m1.pivot(index='metric', columns='model', values='relative_change_pct').reindex(metrics))
display(screen.timing_summary(prepared))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 5. 결과 ZIP 백업·다운로드

CSV/JSON·원본 완료arm·학습이력·시간계측을 ZIP에 담습니다. 큰 checkpoint는 ZIP에 넣지 않고 Drive의 progress 폴더에 보존합니다. Drive 결과 폴더에도 ZIP을 복사하므로 브라우저 다운로드가 멈추면 Drive에서 직접 받을 수 있습니다.

완료된 일부 모델만 회수할 때 `ALLOW_PARTIAL=True`로 지정하세요. 이는 학습 조건이나 평가 시점을 바꾸지 않습니다. 아직 학습 중인 셀을 두고 이 셀을 실행하지 마세요.

In [ ]:
from google.colab import files
import shutil, hashlib

ALLOW_PARTIAL = False
output = Path(screen.export_zip(prepared,
    '/content/hm_lightgcn_final8_lastweek_seed51_results.zip', allow_partial=ALLOW_PARTIAL))
destination = Path(cfg.out_dir) / output.name
shutil.copy2(output, destination)
assert hashlib.sha256(output.read_bytes()).digest() == hashlib.sha256(destination.read_bytes()).digest()
print('Drive ZIP 저장 완료:', destination)
print(f'크기: {output.stat().st_size / 1024**2:.2f} MB')
files.download(str(output))
